# 01 · Reconstruction: the combined development index

The descriptive *past* layer of Amber. It takes the tidy panel the data layer wrote, scores every indicator against fixed goalposts, and aggregates the scores into three pillar sub-indices and one combined index.

This notebook is a walkthrough, not an implementation. Every step calls into `src/amber`, and `make index` produces the same table and charts from the command line.

**Prerequisite:** `make panel` (writes `data/processed/panel_interpolated.csv`).

In [ ]:
%matplotlib inline
import logging

import pandas as pd

from amber import config, figures, reconstruction
from amber.modeling import index

logging.basicConfig(level=logging.WARNING)
pd.set_option("display.precision", 3)

panel = reconstruction.load_panel(
    config.PROCESSED_DATA_DIR / f"{config.PANEL_INTERPOLATED_STEM}.csv"
)
panel.shape

## 1 · Normalize

Each indicator is rescaled to [0, 1] against **fixed goalposts**: a low and a high bound per indicator, set in `config.py` and never recomputed. Every country-year sits on the same ruler, whether it's historical, a counterfactual or a projection. Adding a country or a data vintage can't change a past score. The rules, all from `config.py`:

- **Goalposts:** HDI and SDG standards where they exist. Otherwise seeded once from the 2011–2024 range, padded by 25% of its span, then frozen.
- **Index indicators:** nine of the panel's eleven. Poverty and high-tech exports stay in the panel as history, but `config.INDEX_EXCLUDED` keeps them out of the index. No counterfactual or projection can produce them, and the index has to mean the same thing in every layer.
- **Polarity:** under-5 mortality is lower-is-better, so it is inverted: `(high − x) / (high − low)`.
- **Log income:** GDP per capita is taken as `ln(x)` first, following the HDI. An extra dollar matters more at $1,000 per head than at $5,000.
- **Floor:** scores are clipped to [0.01, 1], so a geometric mean can't collapse to 0.

Missing values are dropped, not scored, so "missing" never gets confused with "worst".

In [ ]:
pd.DataFrame(
    {
        ind: {"low": g.low, "high": g.high, "source": g.source[:70]}
        for ind, g in config.GOALPOSTS.items()
    }
).T

In [ ]:
normalized = index.normalize_indicators(panel)

(
    normalized.query("country_iso3 == 'MMR' and year == 2019")
    .assign(polarity=lambda d: d.indicator_id.map(config.INDICATOR_POLARITY).astype(str))[
        ["pillar", "indicator_id", "polarity", "value", "normalized"]
    ]
    .sort_values(["pillar", "indicator_id"])
)

## 2 · Pillars and the combined index

- **Pillar sub-index:** the geometric mean of the normalized indicators observed in that pillar for that country-year.
- **Combined index:** `exp(Σ wᵢ · ln pᵢ / Σ wᵢ)` across the three pillars. The weights are equal by default and renormalized, so any scale works.

A geometric mean stops a strong pillar from covering for a weak one. For the same reason, a country-year only gets a combined score when every weighted pillar is present.

In [ ]:
idx = index.compute_index(panel)

idx.query("series == 'combined'").pivot(index="year", columns="country_name", values="value")

## 3 · Coverage: read this before the charts

A pillar is computed over the indicators that are actually observed. `coverage` is the share observed. When coverage changes, the pillar is built from a different set of indicators, and part of any movement comes from that change rather than from development.

Myanmar's case matters most here:

- All nine index indicators are reported for 2011–2018.
- Secondary enrollment stops after 2018.
- Internet use stops after 2020.
- Health expenditure is missing for 2024, so the 2024 score rests on six of nine.

The charts draw every partial-coverage point as a hollow ring. The future layer (notebook 03) measures what the gaps do to Myanmar's 2024 score.

In [ ]:
idx.query("country_iso3 == 'MMR' and series != 'combined'").pivot(
    index="year", columns="series", values="coverage"
)

## 4 · Charts

The same three figures `make index` writes to `reports/figures/`.

In [ ]:
figures.combined_index_figure(idx)

In [ ]:
figures.myanmar_pillars_figure(idx)

Real GDP per capita, uncomposited. On a log scale, equal slopes mean equal growth rates. WDI reports Myanmar on its October–September fiscal year, so 2020 and 2021 do not line up with calendar-year IMF figures. That is a known difference between the two bases. It is not a data error.

In [ ]:
figures.gdp_pc_divergence_figure(panel)

## 5 · The weights are a setting

How development is defined is a choice. The same data under a different weighting:

In [ ]:
weightings = {
    "equal": None,
    "economy-led": {"economy": 2, "innovation": 1, "human_development": 1},
    "people-led": {"economy": 1, "innovation": 1, "human_development": 2},
}

pd.DataFrame(
    {
        name: index.compute_index(panel, w)
        .query("series == 'combined' and year == 2019")
        .set_index("country_name")["value"]
        for name, w in weightings.items()
    }
).sort_values("equal", ascending=False)